[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/InterstellarNerd8760/lunar-atlas/blob/proxima/terrain-notebooks/notebooks/prepare-landing-sites.ipynb)

# Landing sites: ground patch + horizon for every site we have data for

Goal: zoom from the globe down to standing at a landing site and look around. For each site in `public/data/landing-sites.csv` this notebook makes:
- **ground**: a 12 km square at the sharpest terrain that covers it (PGDA 5 m south-pole maps, LOLA polar grids, SLDEM2015, or LOLA 512 px/deg).
- **horizon**: a 200 km square at 512 px, so far mountains show on the skyline.
- **nac** (optional): a close-up from an LROC NAC stereo terrain model (2 to 5 m/px) where one covers the site.

Every grid is centered on the site (x east, y north, in meters), so the landing point is always the middle pixel. Same float32 + JSON format as `public/data/global-height`. Sites with no coordinates or no data are skipped and listed in `index.json`.

Free CPU runtime on Colab or Kaggle (Kaggle: turn on **Internet**). **Runtime → Run all.** All sites take roughly 20 to 40 minutes.

In [ ]:
%pip -q install rasterio matplotlib requests

## 1. Settings

In [ ]:
SITE_IDS = None          # None = every site in the CSV, or a list like ["apollo-11", "im-2"]
GROUND_KM = 12           # ground patch width in km
GROUND_MAX_PX = 1024     # cap; coarser sources get fewer pixels (no fake detail)
HORIZON_KM = 200         # horizon box width in km
HORIZON_PX = 512         # 200 km / 512 px is about 390 m/px
INCLUDE_NAC = True       # add the LROC NAC close-up where one exists
NAC_KM = 2               # close-up width in km (2 km at 2 m/px = 1000 px)
NAC_MAX_PX = 1024
KEEP_RAW = False         # PGDA files (40-140 MB) are downloaded; False deletes them after use
SITES_CSV = "public/data/landing-sites.csv"   # used if you run inside the repo
REPO_RAW = "https://raw.githubusercontent.com/InterstellarNerd8760/lunar-atlas"
BRANCHES = ["main", "proxima/terrain-notebooks"]  # where to fetch the CSV from otherwise
ASSET_NAME = "landing-sites"
USE_DRIVE = False        # True: save exports to Google Drive (Colab only)
DOWNLOAD_ZIP = True      # Colab: download the zip at the end

## 2. Setup
Helper functions. Nothing to change here.

In [ ]:
import json
import sys
import time
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rasterio.warp
import requests
from rasterio.enums import Resampling
from rasterio.windows import from_bounds

IN_COLAB = "google.colab" in sys.modules

# Raw downloads always stay on the runtime disk. Only finished exports go to Drive.
RAW = Path("data/raw")
OUT_ROOT = Path("data/work")
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/lunar-atlas")
EXPORT = OUT_ROOT / "exports" / ASSET_NAME
RAW.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)

MOON_RADIUS_M = 1737400
# Moon lat/lon (same sphere as the global-height export).
MOON_GEOGRAPHIC = rasterio.crs.CRS.from_proj4("+proj=longlat +R=1737400 +no_defs")


def remote_size(url):
    """Ask the server how big a file is (HTTP HEAD) without downloading it."""
    response = requests.head(url, allow_redirects=True, timeout=30)
    response.raise_for_status()
    return int(response.headers.get("Content-Length", 0))


def download(url, name=None):
    """Download once into data/raw. Re-running the cell skips finished files."""
    path = RAW / (name or Path(url).name)
    if path.exists():
        return path
    temporary_path = path.with_suffix(path.suffix + ".part")
    with requests.get(url, stream=True, timeout=(15, 120)) as response:
        response.raise_for_status()
        with temporary_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
    temporary_path.replace(path)
    return path


def save_grid(elevation, metadata, folder, name):
    """Write <name>.f32 (float32 little-endian, row-major) and <name>.json."""
    elevation = np.asarray(elevation, dtype="<f4")
    if not np.isfinite(elevation).any():
        raise ValueError(f"{name}: no valid elevations found.")
    folder.mkdir(parents=True, exist_ok=True)
    metadata = dict(metadata)
    metadata["minElevationM"] = float(np.nanmin(elevation))
    metadata["maxElevationM"] = float(np.nanmax(elevation))
    elevation.tofile(folder / f"{name}.f32")
    (folder / f"{name}.json").write_text(json.dumps(metadata, indent=2) + "\n")
    return metadata


def zip_folder(folder, zip_path):
    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as bundle:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, path.relative_to(folder.parent))
    return zip_path


def folder_size_mb(folder):
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) / 1e6

print("Colab" if IN_COLAB else "Not Colab (Kaggle or local)", "| exports go to", EXPORT.resolve())

## 3. Site list and terrain sources
Coordinates come only from NASA's NSSDCA spacecraft table (LROC-imaged positions for almost every lander). NASA did not publish center points for the Artemis III regions, so those rows use the center of the matching PGDA 5 m map where one exists, and the rest are skipped.

In [ ]:
import csv
import os
import warnings
from affine import Affine
from rasterio.crs import CRS
from rasterio.warp import reproject, transform as warp_transform

os.environ.update(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="4",
                  GDAL_HTTP_RETRY_DELAY="3", VSI_CACHE="TRUE")
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

# --- The site list (one CSV in the repo) ---
csv_path = Path(SITES_CSV)
if not csv_path.exists():
    csv_path = RAW / "landing-sites.csv"
    for branch in BRANCHES:
        response = requests.get(f"{REPO_RAW}/{branch}/public/data/landing-sites.csv", timeout=30)
        if response.ok:
            csv_path.write_bytes(response.content)
            print("Site list from branch", branch)
            break
    else:
        raise FileNotFoundError("Could not find public/data/landing-sites.csv")
sites = list(csv.DictReader(csv_path.open()))
if SITE_IDS:
    sites = [s for s in sites if s["id"] in SITE_IDS]
print(len(sites), "sites in the list")

# --- Terrain sources, best first. Every URL comes from an official NASA/PDS/LROC page. ---
LOLA = "https://imbrium.mit.edu/DATA/LOLA_GDR"
SLDEM = "https://imbrium.mit.edu/DATA/SLDEM2015/TILES/JP2"
PGDA_5M = "https://pgda.gsfc.nasa.gov/data/LOLA_5mpp"
NAC = "https://pds.lroc.im-ldi.com/data/LRO-L-LROC-5-RDR-V1.0/LROLRC_2001/DATA/SDP/NAC_DTM"
DEG_M = MOON_RADIUS_M * np.pi / 180   # meters per degree of latitude
SOUTH_POLAR = CRS.from_proj4("+proj=stere +lat_0=-90 +lat_ts=-90 +lon_0=0 +R=1737400 +units=m +no_defs")
LON_0_360 = CRS.from_proj4("+proj=longlat +R=1737400 +lon_wrap=180 +no_defs")  # for files that use 0..360 E

PGDA_FOOTPRINTS = {  # x/y box in meters (south polar stereographic), read from each file's header
    "DM1": (154000, 51500, 174000, 71500),
    "DM2": (147000, 85000, 167000, 105000),
    "Haworth": (-52900, 75600, -23100, 105400),
    "LM1": (2000, -2000, 22000, 18000),
    "LM2": (2000, 47000, 22000, 67000),
    "LM3": (4000, 37000, 24000, 57000),
    "LM4": (9000, 15000, 29000, 35000),
    "LM5": (30000, 1000, 50000, 21000),
    "LM6": (49000, 10000, 69000, 30000),
    "LM7": (54000, -10000, 74000, 10000),
    "LM8": (60000, 30000, 80000, 50000),
    "NPA": (-98500, 19500, -78500, 39500),
    "NPB": (175000, -8500, 195000, 11500),
    "NPC": (122500, -109500, 142500, -89500),
    "NPD": (13000, 146000, 33000, 166000),
    "SL2": (-58500, 13000, -38500, 33000),
    "SL3": (-38700, -16100, -18700, 3900),
    "Shoemaker": (66000, 29000, 86000, 49000),
    "Site01": (-19000, -20000, -3000, -4000),
    "Site04": (-9000, -15000, 7000, 1000),
    "Site06": (74000, 100000, 94000, 120000),
    "Site07": (22000, -28000, 38000, -12000),
    "Site11": (-45000, 7000, -29000, 23000),
    "Site20": (65000, 110000, 81000, 126000),
    "Site20v2": (60000, 105000, 86000, 131000),
    "Site23": (-11000, 111000, 10000, 132000),
    "Site42": (-123400, -66100, -103400, -46100),
}


def lola(path, res_m, name=None):
    return {"name": name or Path(path).stem, "url": f"{LOLA}/{path}", "resM": res_m,
            "scale": 0.5, "nodata": None,
            "label": f"{LOLA}/{path}" if path.endswith(".LBL") else f"{LOLA}/{path[:-4]}_JP2.LBL"}

# South-polar LOLA grids (polar stereographic), sharpest first.
POLAR_GROUND = [lola("POLAR/IMG/LDEM_875S_5M.LBL", 5), lola("POLAR/IMG/LDEM_85S_10M.LBL", 10),
                lola("POLAR/IMG/LDEM_80S_20M.LBL", 20), lola("POLAR/IMG/LDEM_75S_30M.LBL", 30),
                lola("POLAR/IMG/LDEM_60S_60M.LBL", 60)]
HORIZON_SOURCES = [lola("POLAR/IMG/LDEM_80S_80M.LBL", 80), lola("POLAR/IMG/LDEM_75S_120M.LBL", 120),
                   lola("POLAR/IMG/LDEM_60S_240M.LBL", 240),
                   dict(lola("CYLINDRICAL/IMG/LDEM_64.LBL", round(DEG_M / 64, 1)), wraps=True)]


def sldem_tile(lat, lon):
    """SLDEM2015 512 px/deg tile (LOLA + Kaguya TC, 60S-60N) that holds this point."""
    if abs(lat) >= 60:
        return None
    bands = [(-60, -30, "60S_30S"), (-30, 0, "30S_00S"), (0, 30, "00N_30N"), (30, 60, "30N_60N")]
    band = next(b for s, n, b in bands if s <= lat < n)
    west = int((lon % 360) // 45 * 45)
    name = f"SLDEM2015_512_{band}_{west:03d}_{west + 45:03d}"
    return {"name": name, "url": f"{SLDEM}/{name}.JP2", "resM": round(DEG_M / 512, 1), "scale": 0.5,
            "nodata": -32768, "label": f"{SLDEM}/{name}_JP2.LBL"}


def ldem512_tile(lat, lon):
    """LOLA 512 px/deg tile (any latitude), used when SLDEM does not cover the box."""
    bands = [(-90, -45, "90S_45S"), (-45, 0, "45S_00S"), (0, 45, "00N_45N"), (45, 90.01, "45N_90N")]
    band = next(b for s, n, b in bands if s <= lat < n)
    west = int((lon % 360) // 90 * 90)
    name = f"LDEM_512_{band}_{west:03d}_{west + 90:03d}"
    return lola(f"CYLINDRICAL/IMG/{name}.LBL", round(DEG_M / 512, 1), name)


def pgda_sources(lat, lon, half_m):
    """PGDA 5 m site DEMs whose footprint holds the whole box."""
    (x,), (y,) = warp_transform(MOON_GEOGRAPHIC, SOUTH_POLAR, [lon], [lat])
    out = []
    for site_id, (left, bottom, right, top) in PGDA_FOOTPRINTS.items():
        if left <= x - half_m and x + half_m <= right and bottom <= y - half_m and y + half_m <= top:
            out.append({"name": site_id, "url": f"{PGDA_5M}/{site_id}/{site_id}_final_adj_5mpp_surf.tif",
                        "resM": 5.0, "scale": 1.0, "nodata": None, "download": True,
                        "label": "https://pgda.gsfc.nasa.gov/products/78"})
    return out


def ground_sources(lat, lon):
    found = pgda_sources(lat, lon, GROUND_KM * 500) if lat < -80 else []
    if lat < -55:
        found += POLAR_GROUND
    found += [s for s in (sldem_tile(lat, lon), ldem512_tile(lat, lon)) if s]
    return found + [HORIZON_SOURCES[-1]]   # global LDEM_64 as the last resort

## 4. How one layer is made
Pick a source, check it covers the whole box, then resample it onto a grid centered on the site. Shrinking uses `average`; enlarging uses `bilinear`.

In [ ]:
class NotCovered(Exception):
    pass


def open_source(source):
    """Local file for PGDA (their server blocks many small reads), streaming for the rest."""
    if source.get("download"):
        return str(download(source["url"]))
    if source["nodata"] is not None:
        return f'vrt:///vsicurl/{source["url"]}?a_nodata={source["nodata"]}'
    return f'/vsicurl/{source["url"]}'


def with_retries(action, tries=3):
    for attempt in range(tries):
        try:
            return action()
        except rasterio.errors.RasterioIOError as error:
            if "404" in str(error) or attempt == tries - 1:
                raise
            time.sleep(5 * (attempt + 1))


def make_layer(source, lat, lon, km, max_px, fixed_px=None, max_nan=0.02):
    """Resample a source onto a square grid centered on the site: x east, y north, meters."""
    site_crs = CRS.from_proj4(f"+proj=aeqd +lat_0={lat} +lon_0={lon} +R=1737400 +units=m +no_defs")
    # Check the source covers the whole box before reading anything big.
    edge = np.linspace(-km * 500, km * 500, 21)
    xs, ys = np.meshgrid(edge, edge)

    def read_wrapping(dataset, px, grid):
        """Whole-Moon grid: read across the 0/360 seam by wrapping columns."""
        b, (width, height) = dataset.bounds, (dataset.width, dataset.height)
        dlon, dlat = dataset.res[0] / DEG_M, dataset.res[1] / DEG_M
        lon_left = dataset.crs.to_dict().get("lon_0", 0) + b.left / DEG_M
        lat_top = b.top / DEG_M
        geo = CRS.from_proj4(f"+proj=longlat +R=1737400 +lon_wrap={lon} +no_defs")
        lons, lats = warp_transform(site_crs, geo, xs.ravel().tolist(), ys.ravel().tolist())
        lon_left += 360 * np.floor((lon - lon_left) / 360)   # so the site falls inside the first copy
        c0, c1 = int(np.floor((min(lons) - lon_left) / dlon)) - 2, int(np.ceil((max(lons) - lon_left) / dlon)) + 2
        r0, r1 = int(np.floor((lat_top - max(lats)) / dlat)) - 2, int(np.ceil((lat_top - min(lats)) / dlat)) + 2
        if r0 < 0 or r1 > height:
            raise NotCovered(f'{source["name"]} does not cover the box')
        cols = np.arange(c0, c1) % width
        pieces, start = [], 0
        while start < len(cols):   # one read per unbroken run of columns
            stop = start + 1
            while stop < len(cols) and cols[stop] == cols[stop - 1] + 1:
                stop += 1
            window = rasterio.windows.Window(int(cols[start]), r0, stop - start, r1 - r0)
            pieces.append(dataset.read(1, window=window, masked=True, out_dtype="float32").filled(np.nan))
            start = stop
        values = np.hstack(pieces)
        src_transform = Affine(dlon, 0, lon_left + c0 * dlon, 0, -dlat, lat_top - r0 * dlat)
        out = np.full((px, px), np.nan, dtype="float32")
        reproject(values, out, src_transform=src_transform, src_crs=geo, src_nodata=np.nan,
                  dst_transform=grid, dst_crs=site_crs, dst_nodata=np.nan, resampling=Resampling.bilinear)
        return out * source["scale"], site_crs, grid

    def read():
        with rasterio.open(open_source(source)) as dataset:
            src_crs = dataset.crs
            if not src_crs.is_geographic:
                source["resM"] = round(float(dataset.res[0]), 3)   # true pixel size of this file
            px = fixed_px or int(min(max_px, max(64, round(km * 1000 / source["resM"]))))
            res = km * 1000 / px
            grid = Affine(res, 0, -km * 500, 0, -res, km * 500)
            if source.get("wraps"):
                return read_wrapping(dataset, px, grid)
            if src_crs.is_geographic and dataset.bounds.right > 180:
                src_crs = LON_0_360
            sx, sy = warp_transform(site_crs, src_crs, xs.ravel().tolist(), ys.ravel().tolist())
            b = dataset.bounds
            if not (min(sx) >= b.left and max(sx) <= b.right and min(sy) >= b.bottom and max(sy) <= b.top):
                raise NotCovered(f'{source["name"]} does not cover the box')
            # Read just the part we need in one go (2 px margin). If it is much bigger than
            # the output, let GDAL shrink it while reading (fast for JP2 files).
            window = from_bounds(min(sx), min(sy), max(sx), max(sy), dataset.transform)
            window = window.round_offsets(op="floor").round_lengths(op="ceil")
            window = rasterio.windows.Window(window.col_off - 2, window.row_off - 2,
                                             window.width + 4, window.height + 4)
            window = window.intersection(rasterio.windows.Window(0, 0, dataset.width, dataset.height))
            factor = max(1, int(min(window.width, window.height) // (2 * px)))
            shape = (max(1, window.height // factor), max(1, window.width // factor))
            values = dataset.read(1, window=window, out_shape=shape, masked=True,
                                  out_dtype="float32", resampling=Resampling.average)
            src_transform = dataset.window_transform(window) * Affine.scale(window.width / shape[1],
                                                                            window.height / shape[0])
            out = np.full((px, px), np.nan, dtype="float32")
            reproject(values.filled(np.nan), out, src_transform=src_transform, src_crs=src_crs,
                      src_nodata=np.nan, dst_transform=grid, dst_crs=site_crs, dst_nodata=np.nan,
                      resampling=Resampling.average if res > 1.5 * source["resM"] else Resampling.bilinear)
            return out * source["scale"], site_crs, grid

    elevation, site_crs, grid = with_retries(read)
    nan_share = float(np.isnan(elevation).mean())
    if nan_share > max_nan:
        raise NotCovered(f'{source["name"]}: {nan_share:.0%} of the box has no data')
    return elevation, grid, site_crs, nan_share


def layer_metadata(site, source, role, elevation, grid, site_crs, km, nan_share):
    h, w = elevation.shape
    return {
        "source": source["url"], "productLabel": source.get("label"), "sourceProduct": source["name"],
        "siteId": site["id"], "siteName": site["name"], "role": role,
        "width": w, "height": h, "encoding": "float32-le", "units": "m", "order": "row-major",
        "nodataEncoding": "NaN", "referenceRadiusM": MOON_RADIUS_M, "globalExtentDegrees": None,
        "projection": "azimuthal equidistant centered on the site; x east, y north, meters",
        "crsWkt": site_crs.to_wkt(), "affine": list(grid)[:6],
        "extentM": [-km * 500, -km * 500, km * 500, km * 500],
        "centerLatLon": [float(site["lat"]), float(site["lon"])],
        "resolutionM": grid.a, "sourceResolutionM": source["resM"],
        "sourceScale": source["scale"], "sourceOffset": 0.0,
        "siteHeightM": float(elevation[h // 2, w // 2]), "noDataShare": round(nan_share, 4),
        "coordinateSource": site["coord_source"],
    }

## 5. Process every site
Problems are logged and the site is skipped, so one bad file never stops the run.

In [ ]:
index, skipped = [], []
for site in sites:
    sid = site["id"]
    if not site["lat"] or not site["lon"]:
        skipped.append({"id": sid, "name": site["name"], "reason": site["coord_source"]})
        print(f"SKIP {sid}: no coordinates ({site['coord_source']})")
        continue
    lat, lon = float(site["lat"]), float(site["lon"])
    folder = EXPORT / sid
    entry = {"id": sid, "name": site["name"], "mission": site["mission"], "year": int(site["year"]),
             "country_or_company": site["country_or_company"], "lat": lat, "lon": lon, "layers": {}}
    start = time.time()
    try:
        # 1. Ground patch: the sharpest source that covers the whole box.
        problems = []
        for source in ground_sources(lat, lon):
            try:
                elevation, grid, site_crs, nan_share = make_layer(source, lat, lon, GROUND_KM, GROUND_MAX_PX)
                break
            except (NotCovered, rasterio.errors.RasterioIOError) as error:
                problems.append(str(error)[:120])
        else:
            raise NotCovered("no ground source: " + "; ".join(problems))
        meta = save_grid(elevation, layer_metadata(site, source, "ground", elevation, grid, site_crs,
                                                   GROUND_KM, nan_share), folder, f"{sid}-ground")
        entry["layers"]["ground"] = {"file": f"{sid}/{sid}-ground.f32", "source": source["name"],
                                     "resolutionM": meta["resolutionM"], "sizePx": meta["width"]}
        entry["siteHeightM"] = meta["siteHeightM"]
        if source.get("download") and not KEEP_RAW:
            (RAW / Path(source["url"]).name).unlink(missing_ok=True)

        # 2. Horizon box: wider and coarser, for distant ridges.
        last_problem = "no source"
        for source in HORIZON_SOURCES:
            if "POLAR" in source["url"] and lat > -55:
                continue   # polar grids only reach about 60 S
            try:
                elevation, grid, site_crs, nan_share = make_layer(source, lat, lon, HORIZON_KM, None,
                                                                  fixed_px=HORIZON_PX)
                break
            except (NotCovered, rasterio.errors.RasterioIOError) as error:
                last_problem = str(error)[:120]
        else:
            source = None
            print(f"  {sid}: no horizon box ({last_problem})")
        if source:
            meta = save_grid(elevation, layer_metadata(site, source, "horizon", elevation, grid, site_crs,
                                                       HORIZON_KM, nan_share), folder, f"{sid}-horizon")
            entry["layers"]["horizon"] = {"file": f"{sid}/{sid}-horizon.f32", "source": source["name"],
                                          "resolutionM": meta["resolutionM"], "sizePx": meta["width"]}

        # 3. Optional LROC NAC DTM close-up (about 2 to 5 m/px) where one covers the site.
        if INCLUDE_NAC and site["nac_dtm"]:
            name = site["nac_dtm"]
            source = {"name": f"NAC_DTM_{name}", "url": f"{NAC}/{name}/NAC_DTM_{name}.TIF",
                      "label": f"{NAC}/{name}/NAC_DTM_{name}.LBL", "resM": 2.0,
                      "scale": 1.0, "nodata": None}
            try:
                elevation, grid, site_crs, nan_share = make_layer(source, lat, lon, NAC_KM, NAC_MAX_PX,
                                                                  max_nan=0.5)
                meta = save_grid(elevation, layer_metadata(site, source, "nac-closeup", elevation, grid,
                                                           site_crs, NAC_KM, nan_share), folder, f"{sid}-nac")
                entry["layers"]["nac"] = {"file": f"{sid}/{sid}-nac.f32", "source": source["name"],
                                          "resolutionM": meta["resolutionM"], "sizePx": meta["width"]}
            except NotCovered as error:
                print(f"  {sid}: NAC DTM skipped ({error})")
        index.append(entry)
        layers = ", ".join(f'{k} {v["sizePx"]}px @ {v["resolutionM"]:.1f} m ({v["source"]})'
                           for k, v in entry["layers"].items())
        print(f"OK   {sid}: {layers}  [{time.time() - start:.0f} s]")
    except Exception as error:   # log and keep going
        skipped.append({"id": sid, "name": site["name"], "reason": f"{type(error).__name__}: {error}"[:300]})
        print(f"SKIP {sid}: {error}")

(EXPORT / "index.json").write_text(json.dumps({
    "description": "Ground patch, horizon box, and optional NAC close-up per landing site",
    "groundKm": GROUND_KM, "horizonKm": HORIZON_KM, "nacKm": NAC_KM,
    "encoding": "float32-le", "units": "m", "nodataEncoding": "NaN", "referenceRadiusM": MOON_RADIUS_M,
    "sites": index, "skipped": skipped,
}, indent=2) + "\n")
print(f"\n{len(index)} sites exported, {len(skipped)} skipped")

## 6. Quick look

In [ ]:
# Quick look at the first few sites: ground patch and horizon box.
for entry in index[:4]:
    fig, axes = plt.subplots(1, len(entry["layers"]), figsize=(4 * len(entry["layers"]), 4))
    for axis, (role, layer) in zip(np.atleast_1d(axes), entry["layers"].items()):
        meta = json.loads((EXPORT / layer["file"].replace(".f32", ".json")).read_text())
        grid = np.fromfile(EXPORT / layer["file"], dtype="<f4").reshape(meta["height"], meta["width"])
        half = meta["extentM"][2] / 1000
        axis.imshow(grid, cmap="terrain", extent=[-half, half, -half, half])
        axis.plot(0, 0, "r+")
        axis.set_title(f'{entry["id"]} {role}\n{meta["resolutionM"]:.1f} m/px')
        axis.set_xlabel("km east")
    plt.show()

## 7. Zip and download
Unzip into `public/data/` to get `public/data/landing-sites/index.json` and one folder per site.

In [ ]:
zip_path = zip_folder(EXPORT, OUT_ROOT / f"{ASSET_NAME}.zip")
print(f"Exports: {folder_size_mb(EXPORT):.1f} MB in {EXPORT}")
print(f"Zip: {zip_path.stat().st_size / 1e6:.1f} MB at {zip_path}")
if DOWNLOAD_ZIP and IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Download the zip from the file panel (Kaggle: Output tab).")